In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Shadipur_Delhi_CPCB_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,270.0,229.0,317.0,117.0,91.0,93.0,47.0,101.0,167.0,151.0,381.0,316.0
1,2,369.0,265.0,354.0,265.0,74.0,122.0,57.0,86.0,177.0,171.0,401.0,255.0
2,3,378.0,231.0,277.0,260.0,112.0,142.0,130.0,84.0,167.0,211.0,439.0,253.0
3,4,356.0,325.0,210.0,120.0,77.0,212.0,152.0,92.0,150.0,274.0,475.0,224.0
4,5,335.0,329.0,288.0,155.0,182.0,176.0,105.0,82.0,146.0,304.0,476.0,203.0
5,6,395.0,322.0,259.0,186.0,258.0,182.0,84.0,90.0,124.0,324.0,415.0,253.0
6,7,398.0,335.0,253.0,196.0,192.0,275.0,93.0,93.0,101.0,320.0,414.0,326.0
7,8,397.0,193.0,291.0,191.0,133.0,299.0,88.0,102.0,101.0,265.0,434.0,331.0
8,9,442.0,280.0,159.0,272.0,223.0,232.0,87.0,NaN,68.0,285.0,459.0,333.0
9,10,401.0,273.0,308.0,300.0,195.0,170.0,78.0,204.0,58.0,268.0,316.0,315.0


In [4]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    36 non-null     float64
 2   February   33 non-null     float64
 3   March      35 non-null     float64
 4   April      34 non-null     float64
 5   May        37 non-null     float64
 6   June       34 non-null     float64
 7   July       35 non-null     float64
 8   August     35 non-null     float64
 9   September  33 non-null     float64
 10  October    36 non-null     float64
 11  November   34 non-null     float64
 12  December   35 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [7]:
# Convert all columns except 'Day' to numeric values
for col in df.columns:
    if col != 'Day':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill missing values with column mean
df_filled = df.fillna(df.mean(numeric_only=True))

In [8]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [10]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,270.0,229.000000,317.000000,117.000000,91.000000,93.000000,47.0,101.000000,167.000000,151.0,381.000000,316.0
1,2,369.0,265.000000,354.000000,265.000000,74.000000,122.000000,57.0,86.000000,177.000000,171.0,401.000000,255.0
2,3,378.0,231.000000,277.000000,260.000000,112.000000,142.000000,79.4,84.000000,167.000000,211.0,439.000000,253.0
3,4,356.0,325.000000,210.000000,120.000000,77.000000,212.000000,79.4,92.000000,150.000000,274.0,475.000000,224.0
4,5,335.0,329.000000,288.000000,155.000000,182.000000,176.000000,105.0,82.000000,146.000000,304.0,476.000000,203.0
5,6,395.0,322.000000,259.000000,186.000000,258.000000,182.000000,84.0,90.000000,124.000000,324.0,415.000000,253.0
6,7,398.0,335.000000,253.000000,196.000000,192.000000,144.235294,93.0,93.000000,101.000000,320.0,414.000000,326.0
7,8,397.0,193.000000,291.000000,191.000000,133.000000,144.235294,88.0,102.000000,101.000000,265.0,434.000000,331.0
8,9,442.0,280.000000,159.000000,272.000000,223.000000,232.000000,87.0,116.942857,68.000000,285.0,459.000000,333.0
9,10,401.0,273.000000,308.000000,300.000000,195.000000,170.000000,78.0,204.000000,58.000000,268.0,316.000000,315.0
